In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn

MODEL_PATH = 'line_follower.pth'
IMG_SIZE   = 224

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# ── Rebuild the same architecture as training ─────────────────────────────────
model = torchvision.models.mobilenet_v2(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 1),  # single output: steering value
    nn.Tanh()                           # clamps output to -1.0 to 1.0
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()
print("Model loaded successfully.")

# Image pre-processing pipeline (must match training)
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

Running on: cuda
Model loaded successfully.


/tmp/ipykernel_2568/3851377811.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=device))


In [3]:
import traitlets
import cv2
import numpy as np
import pyzed.sl as sl
import threading
import time
import motors
from traitlets.config.configurable import SingletonConfigurable

# Adjust these to tune performance on specific track
SPEED_FORWARD   = 0.35   # cruise speed on straight sections
SPEED_TURN      = 0.30   # speed during turns (slightly slower)
SPEED_SEARCH    = 0.25   # spin speed when line is lost
CONF_THRESHOLD  = 0.4   # minimum confidence to trust a prediction
LOST_FRAMES_MAX = 4      # frames of low confidence before declaring line lost

class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super(Camera, self).__init__()
        self.zed = sl.Camera()
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA
        init_params.depth_mode = sl.DEPTH_MODE.NONE
        init_params.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print("Camera Open:", repr(status))
            self.zed.close()
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        camera_info = self.zed.get_camera_information()
        self.width  = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        self.image  = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                bgra = self.image.get_data()
                self.color_value = cv2.cvtColor(bgra, cv2.COLOR_BGRA2BGR)

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()

def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

camera = Camera()
camera.start()

robot = motors.MotorsYukon(mecanum=False)
print("Camera and robot ready.")

[2026-04-30 11:38:41 UTC][ZED][INFO] Logging level INFO
[2026-04-30 11:38:41 UTC][ZED][INFO] Logging level INFO
[2026-04-30 11:38:41 UTC][ZED][INFO] Logging level INFO
[2026-04-30 11:38:42 UTC][ZED][INFO] [Init]  Depth mode: NONE
[2026-04-30 11:38:42 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2026-04-30 11:38:42 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2026-04-30 11:38:42 UTC][ZED][INFO] [Init]  Video mode: VGA@100
[2026-04-30 11:38:42 UTC][ZED][INFO] [Init]  Serial Number: S/N 32709812
Camera and robot ready.


In [4]:
import ipywidgets as widgets
from IPython.display import display
from collections import deque
import numpy as np

# Display widgets
display_feed = widgets.Image(format='jpeg', width='45%')
display_mask = widgets.Image(format='jpeg', width='45%')
status_label = widgets.Label(value='Status: Starting...')
layout = widgets.Layout(width='100%')
display(widgets.VBox([
    widgets.HBox([display_feed, display_mask], layout=layout),
    status_label
]))

YELLOW_LOWER = np.array([20, 100, 100])
YELLOW_UPPER = np.array([35, 255, 255])

# ── Speed parameters ──────────────────────────────────────────────────────────
SPEED_MAX          = 0.85   # max forward speed on straights
SPEED_MIN          = 0.20   # min forward speed mid corner
SPEED_SEARCH       = 0.30   # spin speed when line lost
LOST_FRAMES_MAX    = 15
CORNER_DEPTH_THRESHOLD = 600  # mm

# ── Steering parameters ───────────────────────────────────────────────────────
# Deadzone: steering values within this range of 0.0 are treated as straight
STEERING_DEADZONE  = 0.20   # increase to reduce corrections on straights
# Gain: scales how aggressively the steering value maps to turn speed
STEERING_GAIN      = 0.4   # turn_speed = STEERING_GAIN * abs(steering)

# ── Smoothing ─────────────────────────────────────────────────────────────────
steering_history  = deque(maxlen=4)   # smooth steering over last 4 frames
lost_frame_count  = 0
last_steering     = 0.0
frame_count       = 0

def predict_steering(frame):
    """Returns a steering value from -1.0 (full left) to 1.0 (full right)."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    x   = preprocess(rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(x)
    return out.item()   # single float between -1.0 and 1.0

def yellow_mask_debug(frame):
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, YELLOW_LOWER, YELLOW_UPPER)
    return cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)

def is_corner_approaching(depth_image):
    if depth_image is None:
        return False
    h, w = depth_image.shape
    strip = depth_image[int(h * 0.6):int(h * 0.85), int(w * 0.3):int(w * 0.7)]
    strip = np.nan_to_num(strip, nan=0.0).astype(np.float32)
    strip[strip < 100]  = 0
    strip[strip > 3000] = 0
    if strip[strip != 0].size == 0:
        return False
    return strip[strip != 0].min() < CORNER_DEPTH_THRESHOLD

def on_frame(change):
    global lost_frame_count, last_steering, frame_count

    frame = change['new']
    if frame is None:
        return

    frame_count += 1

    # ── Steering prediction ───────────────────────────────────────────────────
    raw_steering = predict_steering(frame)

    # Smooth steering over last N frames
    steering_history.append(raw_steering)
    steering = sum(steering_history) / len(steering_history)

    # ── Line lost detection ───────────────────────────────────────────────────
    # If steering is saturating at the extremes for many frames, line is lost
    if abs(raw_steering) > 0.90:
        lost_frame_count += 1
    else:
        lost_frame_count = 0
        last_steering    = steering

    line_lost = lost_frame_count >= LOST_FRAMES_MAX

    # ── Corner detection ──────────────────────────────────────────────────────
    corner_near = is_corner_approaching(
        camera.depth_image if hasattr(camera, 'depth_image') else None
    )
    sharp_turn = abs(steering) > 0.55   # model predicts a big correction needed

    # Forward speed reduces as steering magnitude increases
    # This naturally slows down on corners without needing depth
    speed_scale  = 1.0 - (abs(steering) * 0.7)   # 1.0 on straight, ~0.3 at full turn
    fwd_speed    = SPEED_MIN + (SPEED_MAX - SPEED_MIN) * speed_scale
    if corner_near:
        fwd_speed = min(fwd_speed, 0.22)   # hard cap near obstacles

    # ── Motor commands ────────────────────────────────────────────────────────
    if line_lost:
        # Spin in the direction of last known steering
        if last_steering < 0:
            robot.left(SPEED_SEARCH)
        else:
            robot.right(SPEED_SEARCH)
        status = f'SEARCHING  lost={lost_frame_count}f'

    elif abs(steering) < STEERING_DEADZONE:
        # Inside deadzone — go straight, no correction
        robot.forward(fwd_speed)
        status = f'FORWARD {fwd_speed:.2f}  steer={steering:.2f}'

    elif steering < 0:
        # Negative = line is to the left, turn left
        turn_speed = STEERING_GAIN * abs(steering)
        robot.left(round(turn_speed, 3))
        status = f'LEFT {turn_speed:.2f}  steer={steering:.2f}'

    else:
        # Positive = line is to the right, turn right
        turn_speed = STEERING_GAIN * abs(steering)
        robot.right(round(turn_speed, 3))
        status = f'RIGHT {turn_speed:.2f}  steer={steering:.2f}'

    status_label.value = f'Status: {status}'

    # ── Display (every 5th frame only) ───────────────────────────────────────
    if frame_count % 5 == 0:
        annotated = frame.copy()
        h, w      = annotated.shape[:2]

        # Draw steering bar at bottom of frame
        bar_centre = w // 2
        bar_end    = int(bar_centre + steering * (w // 3))
        colour     = (0, 255, 0) if not line_lost else (0, 0, 255)
        cv2.line(annotated, (bar_centre, h - 10), (bar_end, h - 10), colour, 4)
        cv2.circle(annotated, (bar_centre, h - 10), 4, (255, 255, 255), -1)

        # Draw deadzone markers
        dz_left  = int((0.5 - STEERING_DEADZONE / 2) * w)
        dz_right = int((0.5 + STEERING_DEADZONE / 2) * w)
        cv2.line(annotated, (dz_left,  h - 20), (dz_left,  h), (255, 255, 0), 1)
        cv2.line(annotated, (dz_right, h - 20), (dz_right, h), (255, 255, 0), 1)

        if corner_near:
            cv2.putText(annotated, 'CORNER',
                        (10, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)

        cv2.putText(annotated, f'steer: {steering:.2f}',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, colour, 2)
        cv2.putText(annotated, f'speed: {fwd_speed:.2f}',
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, colour, 2)

        scale = 0.3
        display_feed.value = bgr8_to_jpeg(
            cv2.resize(annotated, None, fx=scale, fy=scale))
        display_mask.value = bgr8_to_jpeg(
            cv2.resize(yellow_mask_debug(frame), None, fx=scale, fy=scale))

camera.observe(on_frame, names=['color_value'])
print("Line follower running. Execute the cell below to stop.")

Line follower running. Execute the cell below to stop.


In [5]:
camera.unobserve(on_frame, names=['color_value'])
robot.stop()
print("Robot stopped.")


Robot stopped.


## Step 4 — Fine-Tuning Tips

| Problem | Fix |
|---|---|
| Robot overshoots sharp turns | Reduce `SPEED_TURN` or increase `LOST_FRAMES_MAX` |
| Robot stops too early at turns | Reduce `CONF_THRESHOLD` (e.g. 0.45) |
| Mask shows false yellow detections | Narrow `YELLOW_UPPER[0]` (Hue upper bound) |
| Line not detected outdoors / different lighting | Re-collect data under those conditions |
| Prediction jitters left/right on straights | Collect more `forward` examples near line edges |

In [ ]:
import shutil
import os

shutil.make_archive('dataset_backup', 'zip', '.', 'dataset')
print(f"Done! Size: {os.path.getsize('dataset_backup.zip') / 1024 / 1024:.1f} MB")